In [1]:
print("hey")

hey


In [2]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

True

In [3]:
from langchain_groq import ChatGroq
import os

In [4]:
import requests
import os

api_key = os.environ.get("GROQ_API_KEY")
url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

print(response.json())

{'object': 'list', 'data': [{'model_card': 'https://openai.com/index/gpt-oss-model-card', 'id': 'openai/gpt-oss-120b', 'object': 'model', 'created': 1754408224, 'owned_by': 'OpenAI', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 65536, 'hugging_face_id': 'openai/gpt-oss-120b', 'name': 'GPT OSS 120B', 'input_modalities': ['text'], 'output_modalities': ['text'], 'context_length': 131072, 'max_output_length': 65536, 'pricing': {'prompt': '0.00000015', 'completion': '0.0000006', 'image': '0', 'request': '0', 'input_cache_read': '0.000000075'}, 'supported_sampling_parameters': ['temperature', 'top_p', 'stop', 'seed', 'max_tokens'], 'supported_features': ['tools', 'json_mode', 'structured_outputs', 'reasoning']}, {'model_card': 'https://openai.com/index/gpt-oss-model-card', 'id': 'openai/gpt-oss-20b', 'object': 'model', 'created': 1754407957, 'owned_by': 'OpenAI', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens':

In [5]:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [6]:
llm.invoke("What is the llm?")

AIMessage(content='A **large language model (LLM)** is a type of artificial‑intelligence system that’s been trained on massive amounts of text data to understand and generate human‑like language. Here are the key points:\n\n| Aspect | What It Means |\n|--------|---------------|\n| **Scale** | “Large” refers to both the size of the training dataset (often billions of words) and the number of model parameters (from hundreds of millions to hundreds of billions). |\n| **Architecture** | Most modern LLMs use the **transformer** architecture, which excels at capturing long‑range dependencies in text. |\n| **Training** | The model learns by predicting the next token (word, sub‑word, or character) given the preceding context. This self‑supervised learning doesn’t require labeled data. |\n| **Capabilities** | After training, an LLM can: <br>• Complete sentences or paragraphs <br>• Answer questions <br>• Summarize, translate, or rewrite text <br>• Perform reasoning tasks when prompted appropriat

In [7]:
from langchain.tools import tool

@tool
def search_database(query: str, limit: int = 10) -> str:
    """Search the customer database for records matching the query.

    Args:
        query: Search terms to look for
        limit: Maximum number of results to return
    """
    return f"Found {limit} results for '{query}'"

In [8]:
search_database.invoke({
    "query": "customer name",
    "limit": 5
})

"Found 5 results for 'customer name'"

In [9]:
from langgraph.store.memory import InMemoryStore
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain_openai import ChatOpenAI

# Access memory
@tool
def get_user_info(user_id: str, runtime: ToolRuntime) -> str:
    """Look up user info."""
    store = runtime.store
    user_info = store.get(("users",), user_id)
    return str(user_info.value) if user_info else "Unknown user"

# Update memory
@tool
def save_user_info(user_id: str, name: str, age: int, email: str, runtime: ToolRuntime) -> str:
    """Save user info."""
    store = runtime.store
    store.put(("users",), user_id, {"name": name, "age": age, "email": email})
    return "Successfully saved user info."


store = InMemoryStore()
agent = create_agent(
    llm,
    tools=[get_user_info, save_user_info],
    store=store
)

# First session: save user info
agent.invoke({
    "messages": [{"role": "user", "content": "Save the following user: userid: abc123, name: Foo, age: 25, email: foo@langchain.dev"}]
})



{'messages': [HumanMessage(content='Save the following user: userid: abc123, name: Foo, age: 25, email: foo@langchain.dev', additional_kwargs={}, response_metadata={}, id='14ca01a1-7d69-4998-9d32-67f015b9ce94'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to save user info. Use save_user_info function.', 'tool_calls': [{'id': 'fc_54fcd32e-ad86-4cae-9340-39248ba4bf60', 'function': {'arguments': '{"age":25,"email":"foo@langchain.dev","name":"Foo","user_id":"abc123"}', 'name': 'save_user_info'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 179, 'total_tokens': 249, 'completion_time': 0.147248864, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.009866909, 'prompt_tokens_details': None, 'queue_time': 0.351476103, 'total_time': 0.157115773}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_02b0d31eca', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'lo

In [10]:
# Second session: get user info
agent.invoke({
    "messages": [{"role": "user", "content": "Get user info for user with id 'abc123'"}]
})
# Here is the user info for user with ID "abc123":
# - Name: Foo
# - Age: 25
# - Email: foo@langchain.dev

{'messages': [HumanMessage(content="Get user info for user with id 'abc123'", additional_kwargs={}, response_metadata={}, id='83fa1c44-f259-4636-bcde-7e14b4a12830'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "We need to call get_user_info with user_id 'abc123'.", 'tool_calls': [{'id': 'fc_c49caec9-d54c-441b-b800-c5b4056fc449', 'function': {'arguments': '{"user_id":"abc123"}', 'name': 'get_user_info'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 164, 'total_tokens': 209, 'completion_time': 0.09545325, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.006627896, 'prompt_tokens_details': None, 'queue_time': 0.213080635, 'total_time': 0.102081146}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_69334c0be3', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a18c5d-4145-7fc2-a988-a7a941b0453e-0', tool_calls=[{

In [11]:

import requests
from pydantic import BaseModel, Field
from langchain_core.tools import StructuredTool


class WeatherInput(BaseModel):
    city: str = Field(description="City name")


def get_weather(city: str) -> str:
    """Fetch current weather for a city."""

    # Step 1: Convert city name to latitude and longitude
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"

    geo_response = requests.get(
        geo_url,
        params={"name": city, "count": 1, "language": "en"},
        timeout=10,
    )
    geo_response.raise_for_status()
    places = geo_response.json().get("results", [])

    if not places:
        return f"City not found: {city}"

    place = places[0]
    latitude = place["latitude"]
    longitude = place["longitude"]

    # Step 2: Fetch current weather
    weather_url = "https://api.open-meteo.com/v1/forecast"

    weather_response = requests.get(
        weather_url,
        params={
            "latitude": latitude,
            "longitude": longitude,
            "current": (
                "temperature_2m,"
                "relative_humidity_2m,"
                "apparent_temperature,"
                "wind_speed_10m"
            ),
            "timezone": "auto",
        },
        timeout=10,
    )
    weather_response.raise_for_status()
    current = weather_response.json()["current"]

    return (
        f"Weather in {place['name']}: "
        f"Temperature: {current['temperature_2m']} °C, "
        f"Feels like: {current['apparent_temperature']} °C, "
        f"Humidity: {current['relative_humidity_2m']}%, "
        f"Wind speed: {current['wind_speed_10m']} km/h, "
        f"Observation time: {current['time']}"
    )


weather_tool = StructuredTool.from_function(
    func=get_weather,
    name="get_weather",
    description="Fetch current weather data for a city",
    args_schema=WeatherInput,
)

# Test the tool directly
print(weather_tool.invoke({"city": "Kozhikode"}))


Weather in Kozhikode: Temperature: 29.3 °C, Feels like: 33.0 °C, Humidity: 67%, Wind speed: 10.6 km/h, Observation time: 2026-10-30T16:00


In [12]:

from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[weather_tool],
    system_prompt="You are a helpful weather assistant."
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the weather in Kozhikode?"
        }
    ]
})

print(result["messages"][-1].content)


Here’s the latest weather for **Kozhikode** (as of 2026‑10‑30 16:00):

- **Temperature:** 29.3 °C  
- **Feels like:** 33.0 °C  
- **Humidity:** 67 %  
- **Wind:** 10.6 km/h  

Looks warm and a bit humid, with a gentle breeze. If you’re heading outside, stay hydrated and consider a light, breathable outfit. Let me know if you’d like a forecast for the next few days or any other details!


In [13]:
import requests

class WeatherForecastTool:
    def __init__(self):
        self.api_key = os.environ.get("OPENWEATHERMAP_API_KEY")
        self.base_url = "https://api.openweathermap.org/data/2.5"
        self.geourl="https://api.openweathermap.org/geo/1.0/direct"

    def get_current_weather(self, place:str):
        """Get current weather using coordinates."""

        geo_url=f"{self.geourl}"

        geo_params = {
            "q": place,
            "appid": self.api_key}
        
        geo_response = requests.get(geo_url, params=geo_params, timeout=10)
        geo_response.raise_for_status()
        geo_data = geo_response.json()[0]  # Get the first result
        base_url = f"{self.base_url}/weather"
       
       
        params = {
            "lat": geo_data["lat"],
            "lon": geo_data["lon"],
            "appid": self.api_key,
            "units": "metric",
        }

        response = requests.get(base_url, params=params, timeout=10)
        response.raise_for_status()

        return response.json()
    
    def get_forecast_weather(self, place:str):
        """Get weather forecast of a place"""
        try:
            url = f"{self.base_url}/forecast"
            params = {
                "q": place,
                "appid": self.api_key,
               
                "units": "metric"
            }
            response = requests.get(url, params=params)
            return response.json() if response.status_code == 200 else {}
        except Exception as e:
            raise e

In [14]:
weather_data = WeatherForecastTool().get_current_weather(place="london")
weather_data

{'coord': {'lon': -0.1278, 'lat': 51.5074},
 'weather': [{'id': 800,
   'main': 'Clear',
   'description': 'clear sky',
   'icon': '01d'}],
 'base': 'stations',
 'main': {'temp': 12.85,
  'feels_like': 12.54,
  'temp_min': 12.85,
  'temp_max': 12.85,
  'pressure': 1030,
  'humidity': 90,
  'sea_level': 1030,
  'grnd_level': 1027},
 'visibility': 10000,
 'wind': {'speed': 1.42, 'deg': 69, 'gust': 4.44},
 'clouds': {'all': 0},
 'dt': 1793356274,
 'sys': {'country': 'GB', 'sunrise': 1793343002, 'sunset': 1793378289},
 'timezone': 0,
 'id': 2643743,
 'name': 'London',
 'cod': 200}

In [15]:
forcast_data = WeatherForecastTool().get_forecast_weather(place="london")
forcast_data

{'cod': '200',
 'message': 0,
 'cnt': 40,
 'list': [{'dt': 1793361600,
   'main': {'temp': 12.1,
    'feels_like': 11.74,
    'temp_min': 12.1,
    'temp_max': 13.87,
    'pressure': 1030,
    'sea_level': 1030,
    'grnd_level': 1027,
    'humidity': 91,
    'temp_kf': -1.77,
    'dew_point': 10.68},
   'weather': [{'id': 800,
     'main': 'Clear',
     'description': 'clear sky',
     'icon': '01d'}],
   'clouds': {'all': 9},
   'wind': {'speed': 1.87, 'deg': 82, 'gust': 5.33},
   'visibility': 10000,
   'pop': 0,
   'sys': {'pod': 'd'},
   'dt_txt': '2026-10-30 12:00:00'},
  {'dt': 1793372400,
   'main': {'temp': 13.2,
    'feels_like': 12.74,
    'temp_min': 13.2,
    'temp_max': 14.19,
    'pressure': 1029,
    'sea_level': 1029,
    'grnd_level': 1027,
    'humidity': 83,
    'temp_kf': -0.99,
    'dew_point': 10.38},
   'weather': [{'id': 800,
     'main': 'Clear',
     'description': 'clear sky',
     'icon': '01d'}],
   'clouds': {'all': 5},
   'wind': {'speed': 2.54, 'deg': 9

In [16]:

if forcast_data:
    forecasts = forcast_data.get("list", [])
    forecast_str = f"Weather forecast in london for the next  days:\n"
    for forecast in forecasts:
        date = forecast.get('dt_txt', 'N/A')
        temp = forecast.get('main', {}).get('temp', 'N/A')

        desc = forecast.get('weather', [{}])[0].get('description', 'N/A')
        forecast_str += f"{date}: {temp}°C, {desc}\n"
        
    print(forecast_str)

        
        

Weather forecast in london for the next  days:
2026-10-30 12:00:00: 12.1°C, clear sky
2026-10-30 15:00:00: 13.2°C, clear sky
2026-10-30 18:00:00: 12.08°C, clear sky
2026-10-30 21:00:00: 11.49°C, overcast clouds
2026-10-31 00:00:00: 11.42°C, scattered clouds
2026-10-31 03:00:00: 10.59°C, few clouds
2026-10-31 06:00:00: 9.47°C, overcast clouds
2026-10-31 09:00:00: 10.86°C, broken clouds
2026-10-31 12:00:00: 13.86°C, few clouds
2026-10-31 15:00:00: 13.37°C, clear sky
2026-10-31 18:00:00: 10.5°C, broken clouds
2026-10-31 21:00:00: 8.72°C, overcast clouds
2026-11-01 00:00:00: 8.76°C, broken clouds
2026-11-01 03:00:00: 7.2°C, overcast clouds
2026-11-01 06:00:00: 7.51°C, overcast clouds
2026-11-01 09:00:00: 7.59°C, overcast clouds
2026-11-01 12:00:00: 9.4°C, overcast clouds
2026-11-01 15:00:00: 9.7°C, light rain
2026-11-01 18:00:00: 8.19°C, overcast clouds
2026-11-01 21:00:00: 8.54°C, light rain
2026-11-02 00:00:00: 10.22°C, overcast clouds
2026-11-02 03:00:00: 8.98°C, broken clouds
2026-11-0

In [17]:
from collections import defaultdict

In [18]:
def format_forecast_data(forecast_data):
    """Format the forecast data into a more readable structure."""
    daily = defaultdict(list)
    
    for forecast in forecast_data.get("list", []):
        date = forecast.get('dt_txt', 'N/A').split(' ')[0]  # Get only the date part
        daily[date].append(forecast)

    result = {
        "data": []
    }

    for date, forecasts in daily.items():
        temps = [f.get('main', {}).get('temp', 'N/A') for f in forecasts]
        descriptions = [f.get('weather', [{}])[0].get('description', 'N/A') for f in forecasts]
        description=max(set(descriptions), key=descriptions.count)  # Most common description

        result["data"].append({
            "date": date,
            "min_temp":f'{min(temps) if temps else None}°C',
            "max_temp": f'{max(temps) if temps else None}°C',
            "description": description
        })

    return json.dumps(result, indent=2, ensure_ascii=False)

In [19]:
import json
formated_forecast = format_forecast_data(forcast_data)
print(formated_forecast)

{
  "data": [
    {
      "date": "2026-10-30",
      "min_temp": "11.49°C",
      "max_temp": "13.2°C",
      "description": "clear sky"
    },
    {
      "date": "2026-10-31",
      "min_temp": "8.72°C",
      "max_temp": "13.86°C",
      "description": "few clouds"
    },
    {
      "date": "2026-11-01",
      "min_temp": "7.2°C",
      "max_temp": "9.7°C",
      "description": "overcast clouds"
    },
    {
      "date": "2026-11-02",
      "min_temp": "8.71°C",
      "max_temp": "12.62°C",
      "description": "broken clouds"
    },
    {
      "date": "2026-11-03",
      "min_temp": "7.89°C",
      "max_temp": "12.42°C",
      "description": "clear sky"
    },
    {
      "date": "2026-11-04",
      "min_temp": "8.3°C",
      "max_temp": "8.94°C",
      "description": "few clouds"
    }
  ]
}


In [20]:
formated_forecast = json.loads(formated_forecast)
if formated_forecast:
    forecast_str = "Weather forecast in London for the next 5 days:\n"

    for data in formated_forecast["data"]:
        date = data["date"]
        min_temp = data["min_temp"]
        max_temp = data["max_temp"]
        desc = data["description"]

        forecast_str += (
            f"{date}: Min {min_temp}, Max {max_temp}, "
            f"{desc}\n"
        )

    print(forecast_str)


Weather forecast in London for the next 5 days:
2026-10-30: Min 11.49°C, Max 13.2°C, clear sky
2026-10-31: Min 8.72°C, Max 13.86°C, few clouds
2026-11-01: Min 7.2°C, Max 9.7°C, overcast clouds
2026-11-02: Min 8.71°C, Max 12.62°C, broken clouds
2026-11-03: Min 7.89°C, Max 12.42°C, clear sky
2026-11-04: Min 8.3°C, Max 8.94°C, few clouds



In [21]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

True

In [27]:
import os
import requests

class CurrencyConverterTool:

    def __init__(self):
        self.api_key = os.environ.get("EXCHANGE_RATE_API_KEY")
        self.base_url = "https://api.exchangerate.host/convert"

    def convert_currency(
        self,
        amount: float,
        from_currency: str,
        to_currency: str
    ) -> float:

        params = {
            "access_key": self.api_key,
            "from": from_currency,
            "to": to_currency,
            "amount": amount
        }

        response = requests.get(self.base_url, params=params)
        response.raise_for_status()

        data = response.json()
        print(data)
        if data.get("success"):
            result = data["result"]

            print(
                f"{amount} {from_currency} = "
                f"{result:.2f} {to_currency}"
            )

            print(f"Rate used: {data['info']['quote']}")

            return result

        raise ValueError(data.get("error", {}).get("info", "Conversion failed"))

In [23]:
os.environ.get("EXCHANGE_RATE_API_KEY")

'dafb894a9d899d4cb364e35fadaa9f66'

In [24]:

os.environ.get("OPENWEATHERMAP_API_KEY")

'b91084161a568deda39d9dc764fc7c91'

In [25]:
import os

print(os.getcwd())
print(os.listdir())

d:\real-time-projects\AI-Trip-Planner\notebook
['experiments.ipynb']


In [28]:
CurrencyConverterTool().convert_currency(amount=100.0, from_currency="USD", to_currency="INR")

{'success': True, 'terms': 'https://currencylayer.com/terms', 'privacy': 'https://currencylayer.com/privacy', 'query': {'from': 'USD', 'to': 'INR', 'amount': 100}, 'info': {'timestamp': 1793377385, 'quote': 94.94915}, 'result': 9494.914997}
100.0 USD = 9494.91 INR
Rate used: 94.94915


9494.914997